In [1]:
import pandas as pd
import plotly.express as px

In [2]:
url = "https://raw.githubusercontent.com/HecVelaz/Proyecto-Mburica-/main/03_Datos_obtenidos/Mburicao/Mburicao_Sil.csv"

df0 = pd.read_csv(url, header=None, names=["fecha","nivel"])

In [3]:
inicio = '2025-11-30 23:00:00' ##se adelanta 1 hora luego
fin = '2025-12-31 22:55:00'
mascara = (df0['fecha'] >= inicio) & (df0['fecha'] <= fin)
df_diciembre = df0.loc[mascara].copy()

In [4]:
df_diciembre['fecha'] = pd.to_datetime(df_diciembre['fecha'])
df_diciembre= df_diciembre.sort_values('fecha').reset_index(drop=True)

In [5]:
def round_up_to_next_minute(ts):
    if ts.second != 0 or ts.microsecond != 0:
        # Suma los segundos restantes para llegar al minuto siguiente
        return ts + pd.Timedelta(seconds=(60 - ts.second), microseconds=-ts.microsecond)
    return ts

df_diciembre['fecha'] = df_diciembre['fecha'].apply(round_up_to_next_minute)

In [6]:
# Contar ocurrencias de cada fecha
conteo = df_diciembre['fecha'].value_counts()

# Filtrar aquellas que aparecen más de una vez
repetidas = conteo[conteo > 1]

print(f"Total de fechas con repeticiones: {len(repetidas)}")
print("\nFechas repetidas y su frecuencia:")
print(repetidas)

# Opcional: mostrar las filas correspondientes a esas fechas para inspeccionar
if not repetidas.empty:
    # Crear una lista de las fechas repetidas
    fechas_repetidas = repetidas.index.tolist()
    # Filtrar el DataFrame original por esas fechas y ordenar
    df_repetidas = df_diciembre[df_diciembre['fecha'].isin(fechas_repetidas)].sort_values('fecha')
    display(df_repetidas)

Total de fechas con repeticiones: 91

Fechas repetidas y su frecuencia:
fecha
2025-12-26 00:26:00    33
2025-12-01 06:25:00     2
2025-12-07 08:00:00     2
2025-12-22 04:30:00     2
2025-12-27 05:55:00     2
                       ..
2025-12-05 01:45:00     2
2025-12-03 01:25:00     2
2025-12-06 20:20:00     2
2025-12-27 20:50:00     2
2025-12-27 21:10:00     2
Name: count, Length: 91, dtype: int64


,fecha,nivel
88,2025-12-01 06:25:00,6.578
89,2025-12-01 06:25:00,6.578
199,2025-12-01 15:55:00,6.462
200,2025-12-01 15:55:00,6.462
373,2025-12-02 06:25:00,6.580
...,...,...
7956,2025-12-29 11:50:00,6.467
8158,2025-12-30 04:50:00,6.497
8159,2025-12-30 04:50:00,6.497
8421,2025-12-31 03:00:00,6.499


In [7]:
# =============================================
# ELIMINAR FILAS CON FECHA DUPLICADA (conservar la primera)
# =============================================

# Verificar cuántas filas tenemos antes
print(f"Filas antes de eliminar duplicados: {len(df_diciembre)}")

# Identificar fechas repetidas (opcional, solo para diagnóstico)
duplicados_fecha = df_diciembre.duplicated(subset=['fecha'], keep=False)
print(f"Filas con fecha duplicada (incluyendo primera aparición): {duplicados_fecha.sum()}")
if duplicados_fecha.sum() > 0:
    print("Ejemplo de fechas repetidas (primeras 5):")
    display(df_diciembre[duplicados_fecha].sort_values('fecha').head(10))

# Eliminar duplicados basados en la columna 'fecha', conservando la primera ocurrencia
df_diciembre = df_diciembre.drop_duplicates(subset=['fecha'], keep='first')

# Verificar después
print(f"Filas después de eliminar duplicados: {len(df_diciembre)}")

Filas antes de eliminar duplicados: 8662
Filas con fecha duplicada (incluyendo primera aparición): 213
Ejemplo de fechas repetidas (primeras 5):


,fecha,nivel
88,2025-12-01 06:25:00,6.578
89,2025-12-01 06:25:00,6.578
199,2025-12-01 15:55:00,6.462
200,2025-12-01 15:55:00,6.462
373,2025-12-02 06:25:00,6.580
374,2025-12-02 06:25:00,6.580
553,2025-12-02 21:25:00,6.556
554,2025-12-02 21:25:00,6.556
602,2025-12-03 01:25:00,6.569
603,2025-12-03 01:25:00,6.569


Filas después de eliminar duplicados: 8540


In [8]:
# Contar cuántos duplicados quedan (debe ser 0)
print("Número de filas con fecha duplicada:", df_diciembre.duplicated(subset=['fecha']).sum())

Número de filas con fecha duplicada: 0


In [9]:
df_diciembre['fecha'] = df_diciembre['fecha'] + pd.Timedelta(hours=1)
df_diciembre

,fecha,nivel
0,2025-12-01 00:00:00,6.498
1,2025-12-01 00:05:00,6.503
2,2025-12-01 00:10:00,6.505
3,2025-12-01 00:15:00,6.507
4,2025-12-01 00:20:00,6.511
...,...,...
8657,2025-12-31 23:35:00,6.488
8658,2025-12-31 23:40:00,6.488
8659,2025-12-31 23:45:00,6.489
8660,2025-12-31 23:50:00,6.489


In [10]:
# =============================================
# Crear índice regular de 5 minutos y reindexar
# =============================================

# Asegurar que la columna 'fecha' sea el índice
df_diciembre = df_diciembre.set_index('fecha').sort_index()

# Definir rango completo
start_time = df_diciembre.index.min()
end_time = df_diciembre.index.max()
time_index = pd.date_range(start=start_time, end=end_time, freq='5min')

# Reindexar: las marcas faltantes quedarán con NaN
df_regular = df_diciembre.reindex(time_index)
df_regular.index.name = 'fecha'

# Verificar cuántos NaN se han introducido
print(f"Filas después de reindexar: {len(df_regular)}")
print(f"Valores nulos en 'nivel': {df_regular['nivel'].isna().sum()}")

Filas después de reindexar: 8928
Valores nulos en 'nivel': 389


In [11]:
# Interpolar linealmente los valores nulos en la columna 'nivel'
df_regular['nivel'] = df_regular['nivel'].interpolate(method='linear')

# Verificar que ya no hay nulos
print("Valores nulos después de interpolar:", df_regular['nivel'].isna().sum())

Valores nulos después de interpolar: 0


In [12]:
# =============================================
# APLICAR (7 - nivel)
# =============================================
df_regular['nivel'] = 7 - df_regular['nivel']

In [13]:
# =============================================
# VERIFICAR FRECUENCIA UNIFORME (5 minutos)
# =============================================
diffs = df_regular.index.to_series().diff()
print("\nFrecuencia de las diferencias entre timestamps (debe ser 5 minutos):")
print(diffs.value_counts())


Frecuencia de las diferencias entre timestamps (debe ser 5 minutos):
fecha
0 days 00:05:00    8927
Name: count, dtype: int64


In [14]:
# =============================================
# GRAFICAR INTERACTIVAMENTE CON PLOTLY
# =============================================
import plotly.express as px

fig = px.line(df_regular, x=df_regular.index, y='nivel',
              title='Nivel de agua (m) – diciembre 2025 (corregido +1h, interpolado)')
fig.update_layout(hovermode='x unified')
fig.show()

In [16]:
import pandas as pd
from google.colab import files

df_regular['nivel'] = df_regular['nivel'].round(4)

archivo_excel = '/content/Mburicao_Diciembre_5min.xlsx'
archivo_csv   = '/content/Mburicao_Diciembre_5min.csv'

df_regular.to_excel(archivo_excel, index=True)
df_regular.to_csv(archivo_csv, index=True)

files.download(archivo_excel)
files.download(archivo_csv)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>